# Pricing Your Change — the offline → leaderboard transfer ratio

**What this notebook is for.** You measure a change offline (cross-validation on the visible
split) and it looks great. Then you spend one of your daily submissions to find out what it
was really worth on the leaderboard. That is an expensive way to learn a single number —
unless you can *estimate* it in advance.

This notebook builds that estimator:

1. a **synthetic benchmark** where the "hidden set" is under my control, so I can measure the
   transfer ratio `r = Δleaderboard / Δoffline` for two kinds of change;
2. the **empirical distribution of r** for each kind of change, over many random worlds;
3. a **decision rule**: how many public A/B points you need, and when to stop trusting
   an offline conclusion;
4. the estimator applied to three real measurements (aggregate numbers only — no competition
   data is read or published here).

**Everything runs on text this notebook writes itself.** No dataset, no network, no competition
passage. If it runs, it runs for you too.

| | notebook |
|---|---|
| 1 | [Recovering the Generation Grid](https://www.kaggle.com/code/victorzhong0110/name-that-llm-recovering-the-generation-grid) | winning with a structural prior - the block constraint the 18-model prompt grid implies, and what it is worth offline |
| 2 | [When the Offline Score Lies](https://www.kaggle.com/code/victorzhong0110/name-that-llm-when-the-offline-score-lies) | losing with one - a prior worth +0.105 offline and exactly +0.000 on the leaderboard, and how to find out before you spend a submission |
| 3 | **this one** — the general estimator |

### Other public work on this problem

| | public notebook / dataset | what it gives you |
|---|---|---|
| A | [Name That LLM - Who Wrote This? Style Fingerprints](https://www.kaggle.com/code/victorzhong0110/name-that-llm-who-wrote-this-style-fingerprints) | 81 reproducible style features and the extractor, if you want to *see* the signal this notebook prices |
| B | [Multi-View Features and Public Pool](https://www.kaggle.com/code/victorzhong0110/multi-view-features-public-pool-does-it-work) | when adding views pays off and when it does not - the feature-side half of this notebook, measured on real runs |
| C | [Name That LLM Style Fingerprint Tables](https://www.kaggle.com/datasets/victorzhong0110/name-that-llm-style-fingerprint-tables) | the precomputed feature tables, if you would rather not compute them yourself |
| D | [Name That LLM DeBERTa v3 Base Weights](https://www.kaggle.com/datasets/victorzhong0110/name-that-llm-deberta-v3-base-weights) | the baseline weights, if you want to reproduce the numbers yourself |
| E | [Kaggle Medal Cutoff and Expert Tier Tracker](https://www.kaggle.com/code/victorzhong0110/kaggle-medal-cutoff-and-expert-tier-tracker) | where the bronze line actually sits right now, and how many teams are tied with you on it |

### More public work on this problem

These are the other notebooks and datasets we have published since the table above was
written. They are the same measurements, from different angles:

| | public notebook | what it gives you |
|---|---|---|
| F | [Six Things We Got Wrong About Kaggle Medals](https://www.kaggle.com/code/victorzhong0110/six-things-we-got-wrong-about-kaggle-medals) | six mistakes we made on this account, each one corrected against a measurement - including why a *sum* of votes is not a medal |
| G | [Your Revenue Is Not GMV](https://www.kaggle.com/code/victorzhong0110/your-revenue-is-not-gmv) | a second look at a quantity that is named as revenue but is not realised revenue, on real e-commerce orders |
| H | [ARC-AGI-2 Baseline - Qwen3-4B Test-Time Training](https://www.kaggle.com/code/victorzhong0110/arc-agi-2-qwen3-ttt-baseline) | the price of a full run: test-time training a LoRA per task, DFS beam search over augmentations, and what a 12-hour wall clock buys you |
| I | [VQCC Score and Baselines](https://www.kaggle.com/code/victorzhong0110/vqcc-score-and-baselines) | the official scoring for a quantum-circuit compilation task, and what the published baselines actually score |
| J | [Leak-Safe Feature Toolbox for Tabular Data](https://www.kaggle.com/code/victorzhong0110/leak-safe-feature-toolbox-for-tabular-data) | the feature-side half of this notebook: which transformations are safe to compute before the split, and which are leakage |
| K | [Running Tabular Modeling Recipes on Real Data](https://www.kaggle.com/code/victorzhong0110/running-tabular-modeling-recipes-on-real-data) | the same question at the recipe level: what each standard tabular move is worth on datasets that were not built for it |
| L | [Gemma 4 Submission Pack (flexonafft v10)](https://www.kaggle.com/code/victorzhong0110/gemma4-submission-pack-flexonafft-v10) | a byte-for-byte port of a public gemma-4 submission pack, with the exact bytes verified - the same honesty rule this notebook is built on |

| M | [VQCC Verifier Invariance Audit](https://www.kaggle.com/code/victorzhong0110/vqcc-verifier-invariance-audit) | an audit of the official scorer itself: which parts of a submission can move the score at all, and which are invisible to it |
| N | [VQCC FeMoCo Public Audit](https://www.kaggle.com/code/victorzhong0110/vqcc-femoco-public-audit) | how many factorizations the public FeMoCo package really contains, and how close each one sits to the 20 mHa gate |
| O | [VQCC DF-THC Cost-Precision Atlas](https://www.kaggle.com/code/victorzhong0110/vqcc-df-thc-cost-precision-atlas) | the cheapest feasible word width per system, and the qubit cost cliff |
| P | [Kaggle Medal Cutoff and Expert Tier Tracker](https://www.kaggle.com/code/victorzhong0110/kaggle-medal-cutoff-and-expert-tier-tracker) | what rank actually earns a bronze medal, computed from the official tier rules |
| Q | [VQCC Verifier Invariance Atlas (dataset)](https://www.kaggle.com/datasets/victorzhong0110/vqcc-verifier-invariance-atlas) | the measurements behind notebook M, as three CSV tables |
| R | [VQCC FeMoCo Public Factorization Audit (dataset)](https://www.kaggle.com/datasets/victorzhong0110/vqcc-femoco-public-factorization-audit) | the audit table behind notebook N: 18 factorizations, their residuals and their official costs |

Both are public now, and both link back here.


### Step 7 — the checklist

Before you spend a submission on a change that looked good offline:

1. **Is the change feature-side or structural?** If it assumes a property of the hidden
   *layout* (a grid, an ordering, a label-per-block rule), it is structural — price it with a
   public A/B, not with cross-validation.
2. **Does the offline delta come from a prompt-grouped split?** If the hidden set groups
   differently, the split that flatters you is the wrong one.
3. **Is the probe/feature fitted out-of-fold?** An in-sample linear probe on 768 dense
   features will report a near-perfect score and mean nothing (this notebook's sibling
   measures exactly that gap).
4. **Do you have a knob?** A public A/B is only informative if the two submissions differ by
   one thing.
5. **Do you know your `r̄` for that kind of change yet?** If not, your next submission should
   be the one that buys you that knowledge, not the one that spends it.

### Where to go next

| | notebook |
|---|---|
| 1 | *Recovering the Generation Grid* (not public yet) |
| 2 | *When the Offline Score Lies* (not public yet) |
| 3 | **Pricing Your Change** (this notebook) |

Related public datasets used in this series:
[prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset)
(CC BY-SA 4.0) and
[deberta-v3-base-weights](https://www.kaggle.com/datasets/prvsiyan/deberta-v3-base-weights)
(MIT). This notebook needs neither - it writes its own corpus.


In [ ]:
# ---- Setup (self-contained: no dataset, no network) ------------------------
import os, json, math, time, warnings
from collections import Counter
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.optimize import linear_sum_assignment

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
rng_global = np.random.RandomState(SEED)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
OUTDIR = os.path.join(WORK, "ntl_transfer_ratio")
os.makedirs(OUTDIR, exist_ok=True)
print("workdir:", WORK)


### Step 1 — a synthetic world with a hidden set I control

The point of a synthetic benchmark here is not realism, it is **control**: I decide how far the
hidden set drifts from the visible split, and I decide whether the hidden set still has the
structure my model assumes. Both are knobs, and both are things that happen to you for free
in a real competition.

The generator writes prose. Every passage is assembled from *voices*:

* a **voice** is a bundle of habits — which discourse connectors it reaches for
  (`however`, `moreover`, `therefore`, …), how it opens and closes sentences, its punctuation
  habits (em dash, semicolon), its sentence length, its hedging rate, its topic words;
* an **author** is a **mixture of several voices**, so authors overlap. That overlap is the
  point: if the classes were separable, every model would score ~1.0 and no change could
  show a gain. A real task has headroom, so this one does too;
* a **block** of passages answers one prompt — one passage per author, exactly like a prompt
  grid.


In [ ]:
# ---- The synthetic corpus --------------------------------------------------
CONNECTORS = ["", "", "however", "moreover", "therefore", "indeed", "notably",
              "that said", "in practice", "overall", "furthermore", "essentially"]
OPENERS = ["", "In practice, ", "Overall, ", "Consider ", "That said, ",
           "Broadly, ", "Notably, ", "In short, "]
ENDINGS = ["", " Overall, this matters.", " In practice, the effect is small.",
           " That said, the caveat remains.", " Ultimately, it depends.",
           " Hence the conclusion.", " Still, the trend is clear."]
TOPIC = ["model", "data", "system", "result", "process", "sample", "rate", "value",
         "window", "policy", "record", "signal", "network", "output", "input",
         "balance", "threshold", "estimate", "response", "structure"]
FUNC_WORDS = ["the", "a", "an", "of", "to", "in", "on", "for", "with", "at", "by",
              "from", "as", "is", "are", "was", "were", "be", "and", "or", "but",
              "not", "that", "this", "it", "its", "which", "who", "however",
              "moreover", "therefore", "indeed", "notably", "overall", "furthermore",
              "essentially", "ultimately", "still", "thus", "hence", "so", "if",
              "when", "while", "because", "although", "though", "since", "until",
              "may", "might", "could", "perhaps", "often"]
N_VOICES = 18


def make_voices(rng, n=N_VOICES):
    """A voice = one bundle of writing habits."""
    voices = []
    for _ in range(n):
        voices.append({
            "conn": rng.dirichlet(np.ones(len(CONNECTORS)) * 0.30),
            "open": rng.dirichlet(np.ones(len(OPENERS)) * 0.30),
            "end": rng.dirichlet(np.ones(len(ENDINGS)) * 0.30),
            "slen": int(rng.randint(9, 24)),
            "hedge": float(rng.uniform(0.0, 0.30)),
            "emdash": float(rng.uniform(0.0, 0.14)),
            "semi": float(rng.uniform(0.0, 0.10)),
            "topics": list(rng.choice(len(TOPIC), size=3, replace=False)),
            "topic_bias": float(rng.uniform(0.10, 0.45)),
        })
    return voices


def mix_voices(voices, weights):
    """An author = a mixture of voices (so authors overlap)."""
    w = np.asarray(weights, dtype=float)
    w = w / w.sum()
    out = {}
    for k in ("conn", "open", "end"):
        out[k] = sum(wi * v[k] for wi, v in zip(w, voices))
    out["slen"] = int(round(sum(wi * v["slen"] for wi, v in zip(w, voices))))
    out["hedge"] = float(sum(wi * v["hedge"] for wi, v in zip(w, voices)))
    out["emdash"] = float(sum(wi * v["emdash"] for wi, v in zip(w, voices)))
    out["semi"] = float(sum(wi * v["semi"] for wi, v in zip(w, voices)))
    out["topics"] = sorted(set(t for v in voices for t in v["topics"]))
    out["topic_bias"] = float(sum(wi * v["topic_bias"] for wi, v in zip(w, voices)))
    return out


def make_author_profiles(n_authors, voices, rng, dominant=0.18):
    """Each author is dominated by one voice but carries parts of two others."""
    profiles = []
    for a in range(n_authors):
        picks = rng.choice(len(voices), size=3, replace=False)
        w = np.array([dominant, (1 - dominant) * 0.65, (1 - dominant) * 0.35])
        profiles.append(mix_voices([voices[i] for i in picks], w))
    return profiles


def drift_profile(prof, other, strength, rng):
    """Move an author's habits towards another author's (a hidden-set shift)."""
    mix = lambda a, b: (1 - strength) * np.asarray(a) + strength * np.asarray(b)
    out = dict(prof)
    out["conn"] = mix(prof["conn"], other["conn"])
    out["open"] = mix(prof["open"], other["open"])
    out["end"] = mix(prof["end"], other["end"])
    out["slen"] = int(round((1 - strength) * prof["slen"] + strength * other["slen"]))
    out["hedge"] = (1 - strength) * prof["hedge"] + strength * other["hedge"]
    out["emdash"] = (1 - strength) * prof["emdash"] + strength * other["emdash"]
    out["semi"] = (1 - strength) * prof["semi"] + strength * other["semi"]
    out["topics"] = sorted(set(prof["topics"]) | set(other["topics"]))
    out["topic_bias"] = (1 - strength) * prof["topic_bias"] + strength * other["topic_bias"]
    return out


def write_passage(prof, rng, n_words_target=60):
    """One passage in that author's voice."""
    sents = []
    n_sent = max(3, int(n_words_target / max(prof["slen"], 1)))
    for _ in range(n_sent):
        op = OPENERS[rng.choice(len(OPENERS), p=prof["open"])]
        cn = CONNECTORS[rng.choice(len(CONNECTORS), p=prof["conn"])]
        body_len = max(4, int(prof["slen"] * rng.uniform(0.6, 1.4)))
        body = []
        for _ in range(body_len):
            if rng.rand() < prof["topic_bias"]:
                body.append(TOPIC[rng.choice(prof["topics"])])
            else:
                body.append(TOPIC[rng.choice(len(TOPIC))])
            if rng.rand() < 0.25:
                body.append(rng.choice(["the", "a", "of", "and", "in", "to", "that"]))
            if rng.rand() < prof["hedge"]:
                body.append(rng.choice(["may", "might", "could", "perhaps", "often"]))
        sent = op + (cn + " " if cn else "") + " ".join(body)
        if rng.rand() < prof["emdash"]:
            sent += " --"
        if rng.rand() < prof["semi"]:
            sent += ";"
        sent += ENDINGS[rng.choice(len(ENDINGS), p=prof["end"])]
        sents.append(sent)
    return " ".join(sents)


def make_blocks(n_blocks, profiles, rng):
    """Each block = one prompt = one passage per author."""
    blocks = []
    for b in range(n_blocks):
        rows = []
        for a, prof in enumerate(profiles):
            rows.append({"block": b, "author": a, "text": write_passage(prof, rng)})
        blocks.append(pd.DataFrame(rows))
    return pd.concat(blocks, ignore_index=True)


### Step 2 — two kinds of change, and the two things they assume

**Feature-side change.** Add a new *feature family* to the model. Here: the base model is a
**word n-gram route alone**, and the change adds a **character n-gram route** to the blend —
the same shape as the real competition, where the word route alone scored 0.2719 and adding
the char route took it to 0.3467. Two more views (the **shape** of the text, `Xxxx xx x`, and
**function words only**, every content word replaced by `CONTENT`) are measured alongside, but
they are not the knob. The change assumes only that the hidden passages are prose by the same
authors — no layout, no grouping.

**Structural change.** Add a *constraint* on the output space. Here: **block-constrained
decoding** — inside a block, the K labels must each be used exactly once (a Hungarian
assignment over the block's score matrix). It assumes the hidden set has that exact grid
structure.

Both changes are plausible, both are measured the same way, and only one of them survives
when the hidden set stops looking like the visible split.


In [ ]:
# ---- Routes and the structural decoder --------------------------------------
def route_word_lr():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2,
                                  sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")),
    ])


def route_char_svm():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                                  max_features=100_000, sublinear_tf=True)),
        ("clf", LinearSVC(C=1.0, class_weight="balanced")),
    ])


def func_word_vector(texts, vocab=FUNC_WORDS):
    """Relative frequency of each function word - a route with a different bias
    from TF-IDF, which divides the frequent words away."""
    idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    for r, t in enumerate(texts):
        toks = t.lower().split()
        n = max(len(toks), 1)
        for w in toks:
            j = idx.get(w.strip(",.;:!?()"))
            if j is not None:
                M[r, j] += 1.0
        M[r] /= n
    return M


def route_funcword():
    return Pipeline([
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")),
    ])


import re


def shape_text(t):
    """The SHAPE of the text: Xxxx xx x.

    The word TF-IDF lowercases, so it throws away case habits - which are one of
    the most stable author fingerprints there is. This view puts them back."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def route_shape():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                                  max_features=50_000, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")),
    ])


def fit_scores(train_texts, train_y, test_texts, route, n_classes):
    """Fit a route on train, return class scores on test (n_classes columns)."""
    m = route()
    m.fit(train_texts, train_y)
    if hasattr(m, "predict_proba"):
        p = m.predict_proba(test_texts)
    else:
        from scipy.special import softmax
        p = softmax(m.decision_function(test_texts), axis=1)
    cls = list(m.classes_)
    S = np.zeros((len(test_texts), n_classes))
    for j, c in enumerate(cls):
        S[:, j] = p[:, j]
    return S, cls


def block_constrain(S, groups, classes, valve=True):
    """Assign labels inside each block so that every class is used once.

    `valve=True` only constrains blocks whose size matches the class count -
    the safety valve that turns a wrong prior into ZERO gain instead of
    negative gain (and, importantly, still lets a wrong prior through when
    the block has the right SIZE but the wrong CONTENT)."""
    S = np.asarray(S, dtype=float)
    out = S.copy()
    g = np.asarray(groups)
    n_cls = len(classes)
    applied = 0
    for blk in np.unique(g):
        idx = np.where(g == blk)[0]
        if valve and len(idx) != n_cls:
            continue
        sub = S[idx]
        r, c = linear_sum_assignment(-sub)
        out[idx[r], c] = 1e6            # the assignment wins its rows
        applied += 1
    return out, applied


### Step 3 — the measurement protocol

For each random world:

* **visible split** — `V` blocks. Offline score = prompt-grouped 5-fold (GroupKFold on the
  block id), so no block is ever split across train and validation.
  The base model is the **word route alone**; both changes are measured against that base.
* **hidden split** — `H` blocks, generated with a drift parameter `s` (how far the hidden
  authors have moved) and a **block-structure scenario**.
* **Δoff** = metric(visible, with change) − metric(visible, without change).
* **ΔLB** = metric(hidden, with change) − metric(hidden, without change).
* **r = ΔLB / Δoff**.

The three scenarios for the hidden split:

| scenario | hidden blocks | what it does to the structural prior |
|---|---|---|
| `clean` | complete, one passage per author | the prior is valid |
| `broken` | every block is missing a passage | the prior is invalid, the size valve vetoes it |
| `swapped` | full size, but labels duplicated | the prior is valid in SIZE and wrong in CONTENT |


In [ ]:
# ---- One world: offline delta, hidden delta, transfer ratio -----------------
N_AUTHORS = 8
N_VIS_BLOCKS = 20
N_HID_BLOCKS = 20
W_FEATURE = 0.50          # the one knob of the feature-side change (char route weight)
BLEND_W, BLEND_C = 0.5, 0.5


def _norm(S):
    S = np.asarray(S, dtype=float)
    return S / np.maximum(S.sum(axis=1, keepdims=True), 1e-12)


def run_world(seed, shift=0.35, scenario="clean"):
    rng = np.random.RandomState(seed)
    voices = make_voices(rng)
    profiles = make_author_profiles(N_AUTHORS, voices, rng)

    vis = make_blocks(N_VIS_BLOCKS, profiles, rng)

    # the hidden set: drifted habits, and a block structure I control
    hid_profiles = [drift_profile(p, profiles[(a + 1) % N_AUTHORS], shift, rng)
                    for a, p in enumerate(profiles)]
    hid = make_blocks(N_HID_BLOCKS, hid_profiles, rng)

    if scenario == "broken":
        # EVERY hidden block loses a passage -> the grid constraint is invalid
        keep = [blk.iloc[:-1] for _, blk in hid.groupby("block")]
        hid = pd.concat(keep, ignore_index=True)
    elif scenario == "swapped":
        # right size, wrong content: two authors share a label inside some blocks
        for _, blk in hid.groupby("block"):
            if rng.rand() < 0.5:
                idx = blk.index.to_numpy()
                a = int(rng.randint(0, N_AUTHORS - 1))
                hid.loc[idx[a + 1], "author"] = a
        hid = hid.reset_index(drop=True)

    authors = list(range(N_AUTHORS))
    author_id = {a: i for i, a in enumerate(authors)}
    res = {"seed": int(seed), "shift": shift, "scenario": scenario}

    texts = vis["text"].tolist()
    y = vis["author"].to_numpy()
    groups = vis["block"].to_numpy()
    folds = list(GroupKFold(n_splits=5).split(texts, y, groups))
    FW = func_word_vector(texts)

    # ---------- OFFLINE: prompt-grouped 5-fold on the visible split ----------
    S_word = np.zeros((len(vis), N_AUTHORS))
    S_char = np.zeros((len(vis), N_AUTHORS))
    S_fw = np.zeros((len(vis), N_AUTHORS))
    S_shape = np.zeros((len(vis), N_AUTHORS))
    SHAPE = [shape_text(t) for t in texts]
    for tr, te in folds:
        s, cls = fit_scores([texts[i] for i in tr], y[tr], [texts[i] for i in te],
                            route_word_lr, N_AUTHORS)
        S_word[np.ix_(te, [author_id[c] for c in cls])] = s
        s, cls = fit_scores([texts[i] for i in tr], y[tr], [texts[i] for i in te],
                            route_char_svm, N_AUTHORS)
        S_char[np.ix_(te, [author_id[c] for c in cls])] = s
        m = route_funcword().fit(FW[tr], y[tr])
        p = m.predict_proba(FW[te])
        S_fw[np.ix_(te, [author_id[c] for c in m.classes_])] = p
        m = route_shape().fit([SHAPE[i] for i in tr], y[tr])
        p = m.predict_proba([SHAPE[i] for i in te])
        S_shape[np.ix_(te, [author_id[c] for c in m.classes_])] = p

    # BASE = the word route alone (the weak route).
    # FEATURE-SIDE CHANGE = add the character route to it - the same shape as the
    # real competition, where the word route alone scored 0.2719 and adding the
    # char route took it to 0.3467.
    base_off = _norm(S_word)
    feat_off = _norm((1 - W_FEATURE) * S_word + W_FEATURE * S_char)
    res["offline_base"] = float(balanced_accuracy_score(y, base_off.argmax(1)))
    res["offline_feat"] = float(balanced_accuracy_score(y, feat_off.argmax(1)))
    res["d_off_feature"] = res["offline_feat"] - res["offline_base"]
    res["offline_shape_alone"] = float(balanced_accuracy_score(y, S_shape.argmax(1)))
    res["offline_funcword_alone"] = float(balanced_accuracy_score(y, S_fw.argmax(1)))

    # the STRUCTURAL-side change: block-constrained decoding
    constrained_off, n_applied = block_constrain(base_off, groups, authors)
    res["offline_struct"] = float(balanced_accuracy_score(y, constrained_off.argmax(1)))
    res["d_off_structure"] = res["offline_struct"] - res["offline_base"]
    res["offline_blocks_constrained"] = int(n_applied)

    # ---------- HIDDEN: the "leaderboard" -------------------------------------
    htexts = hid["text"].tolist()
    hy = hid["author"].to_numpy()
    hgroups = hid["block"].to_numpy()
    HFW = func_word_vector(htexts)

    s, cls = fit_scores(texts, y, htexts, route_word_lr, N_AUTHORS)
    H_word = np.zeros((len(hid), N_AUTHORS)); H_word[:, [author_id[c] for c in cls]] = s
    s, cls = fit_scores(texts, y, htexts, route_char_svm, N_AUTHORS)
    H_char = np.zeros((len(hid), N_AUTHORS)); H_char[:, [author_id[c] for c in cls]] = s
    m = route_funcword().fit(FW, y)
    p = m.predict_proba(HFW)
    H_fw = np.zeros((len(hid), N_AUTHORS)); H_fw[:, [author_id[c] for c in m.classes_]] = p
    HSHAPE = [shape_text(t) for t in htexts]
    m = route_shape().fit(SHAPE, y)
    p = m.predict_proba(HSHAPE)
    H_shape = np.zeros((len(hid), N_AUTHORS)); H_shape[:, [author_id[c] for c in m.classes_]] = p

    base_h = _norm(H_word)
    feat_h = _norm((1 - W_FEATURE) * H_word + W_FEATURE * H_char)
    res["hidden_base"] = float(balanced_accuracy_score(hy, base_h.argmax(1)))
    res["hidden_feat"] = float(balanced_accuracy_score(hy, feat_h.argmax(1)))
    res["d_lb_feature"] = res["hidden_feat"] - res["hidden_base"]

    constrained_h, n_applied_h = block_constrain(base_h, hgroups, authors)
    res["hidden_struct"] = float(balanced_accuracy_score(hy, constrained_h.argmax(1)))
    res["d_lb_structure"] = res["hidden_struct"] - res["hidden_base"]
    res["hidden_blocks_constrained"] = int(n_applied_h)

    res["r_feature"] = (res["d_lb_feature"] / res["d_off_feature"]
                         if abs(res["d_off_feature"]) > 1e-9 else np.nan)
    res["r_structure"] = (res["d_lb_structure"] / res["d_off_structure"]
                          if abs(res["d_off_structure"]) > 1e-9 else np.nan)
    return res


### Step 4 — the empirical distribution of r

Run the same protocol over many random worlds and look at where `r` lands.

The headline expectation, and the reason this notebook exists: **the transfer ratio is a
property of the KIND of change, not of how big the offline gain was.** A feature-side change
keeps most of its value when the hidden set drifts; a structural change collapses to zero the
moment the hidden set stops having the structure — even though its offline gain was *larger*.


In [ ]:
# ---- The empirical distribution --------------------------------------------
N_WORLDS = 24
rows = []
for i in range(N_WORLDS):
    scen = ["clean", "broken", "swapped"][i % 3]
    r = run_world(seed=1000 + i, shift=0.35, scenario=scen)
    r["world"] = i
    rows.append(r)
    print("world %2d scen=%-8s base_off=%.3f d_off_feat=%+.4f d_lb_feat=%+.4f r_feat=%+.3f | "
          "d_off_str=%+.4f d_lb_str=%+.4f r_str=%+.3f"
          % (i, scen, r["offline_base"], r["d_off_feature"], r["d_lb_feature"], r["r_feature"],
             r["d_off_structure"], r["d_lb_structure"], r["r_structure"]), flush=True)

W = pd.DataFrame(rows)
print("\n=== transfer ratio r by change type and hidden-set scenario ===")
summary_tab = []
for scen in ["clean", "broken", "swapped"]:
    sub = W[W["scenario"] == scen]
    for kind in ["feature", "structure"]:
        v = sub["r_%s" % kind].dropna().to_numpy()
        if len(v) == 0:
            continue
        summary_tab.append({
            "scenario": scen, "change": kind, "n": int(len(v)),
            "r_mean": float(v.mean()), "r_median": float(np.median(v)),
            "r_p10": float(np.percentile(v, 10)), "r_p90": float(np.percentile(v, 90)),
            "r_min": float(v.min()), "r_max": float(v.max()),
            "d_off_mean": float(sub["d_off_%s" % kind].mean()),
            "d_lb_mean": float(sub["d_lb_%s" % kind].mean()),
        })
TAB = pd.DataFrame(summary_tab)
print(TAB.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, kind, title in ((axes[0], "feature", "feature-side change"),
                        (axes[1], "structure", "structural change")):
    for j, (scen, col) in enumerate((("clean", "#2a9d8f"), ("broken", "#e9c46a"),
                                     ("swapped", "#e76f51"))):
        v = W[W["scenario"] == scen]["r_%s" % kind].dropna().to_numpy()
        ax.scatter(np.arange(len(v)) + (j - 1) * 0.14, v, s=42, alpha=0.85,
                   color=col, label=scen)
    ax.axhline(1.0, ls="--", lw=1, color="#457b9d")
    ax.axhline(0.0, ls="--", lw=1, color="#333333")
    ax.set_title(title)
    ax.set_xlabel("random world")
    ax.set_ylabel("transfer ratio  r = dLB / dOFF")
    ax.set_ylim(-0.6, 2.0)
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "transfer_ratio.png"), dpi=110)
print("wrote", os.path.join(OUTDIR, "transfer_ratio.png"))


### Step 5 — pricing your own change

You cannot measure `r` without spending submissions. But you can **estimate** it, and you can
estimate it from the A/B points you have already paid for.

**The estimator.** For every public A/B you have already paid for — two submissions that differ
by exactly one knob — you have a pair `(Δoff, ΔLB)`. Pool them by *kind of change* and estimate

```
r̂  =  ΔLB / Δoff          (per A/B point)
r̄  =  mean(ΔLB) / mean(Δoff)        (pooled - lower variance)
```

and put a confidence interval on it by bootstrapping over the A/B points. The interval is what
decides the next submission:

* if the **lower bound of r̄ is above your threshold**, the change class is worth buying;
* if it straddles zero, you have learned nothing yet — either get more A/B points of that
  class, or find a cheaper way to test it (a proxy leaderboard, a public split);
* if it is **at or below zero**, stop: the class of change does not survive the hidden set.

**How many A/B points do you need?** The width of the interval shrinks like `1/√n`, and the
simulation below shows the actual numbers: with a realistic spread of `r` you need roughly
**3–5 public A/B points of the same kind** before the sign of `r̄` is stable.


In [ ]:
# ---- The estimator, and how many public A/B points it needs ----------------
def estimate_r(d_off, d_lb, n_boot=20000, seed=0):
    """Bootstrap CI for the pooled transfer ratio of a class of changes."""
    d_off = np.asarray(d_off, dtype=float)
    d_lb = np.asarray(d_lb, dtype=float)
    rng = np.random.RandomState(seed)
    n = len(d_off)
    boots = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        od = d_off[idx].mean()
        if abs(od) < 1e-12:
            continue
        boots.append(d_lb[idx].mean() / od)
    boots = np.asarray(boots)
    return {"n": int(n),
            "r_pooled": float(d_lb.mean() / d_off.mean()),
            "r_mean_of_ratios": float(np.mean(d_lb / d_off)),
            "ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
            "boot_sd": float(boots.std(ddof=1))}


# A simulated history of public A/B points, drawn from the synthetic worlds above.
feat_hist = [(r["d_off_feature"], r["d_lb_feature"]) for _, r in W.iterrows()
             if abs(r["d_off_feature"]) > 1e-6]
struct_hist = [(r["d_off_structure"], r["d_lb_structure"]) for _, r in W.iterrows()
               if abs(r["d_off_structure"]) > 1e-6]

print("=== pooled estimates from the synthetic A/B history ===")
for name, hist in (("feature-side", feat_hist), ("structural-side", struct_hist)):
    if not hist:
        continue
    d_off = [h[0] for h in hist]; d_lb = [h[1] for h in hist]
    est = estimate_r(d_off, d_lb)
    print("%-16s n=%2d  r_pooled=%+.3f  95%% CI [%+.3f, %+.3f]"
          % (name, est["n"], est["r_pooled"], est["ci95"][0], est["ci95"][1]))

print("\n=== how many A/B points until the sign is stable? ===")
print("(the population below is the MEASURED r distribution of the worlds above,")
print(" so this is not a made-up prior - it is the spread the synthetic world produced)")
rng = np.random.RandomState(7)
for kind in ("feature", "structure"):
    emp = W["r_%s" % kind].dropna().to_numpy()
    if len(emp) < 2:
        continue
    print("\n-- %s-side change: %d measured worlds, r spread [%.2f, %.2f]"
          % (kind, len(emp), emp.min(), emp.max()))
    for n in (1, 2, 3, 5, 8, 12):
        draws = rng.choice(emp, size=(20000, n)).mean(axis=1)
        lo, hi = np.percentile(draws, [2.5, 97.5])
        print("   n=%2d A/B points -> mean r = %+.3f, 95%% CI [%+.3f, %+.3f], "
              "P(lower bound <= 0) = %.3f"
              % (n, draws.mean(), lo, hi, float((draws <= 0).mean())))


### Step 6 — the same estimator on three real measurements

These are **aggregate numbers only**: for each change, the offline delta (prompt-grouped
cross-validation on the visible split) and the public-leaderboard delta of one knob. No
passage, prompt, label or row of the underlying competition appears in this notebook — the
three pairs below were already published in
[When the Offline Score Lies](https://www.kaggle.com/code/victorzhong0110/name-that-llm-when-the-offline-score-lies).

| # | change | kind | Δoff | ΔLB | r |
|---|---|---|---|---|---|
| 1 | add a word n-gram route to the blend | feature-side | +0.056 | +0.025 | **0.45** |
| 2 | add a frozen-encoder linear probe | feature-side | +0.0486 | +0.045 | **0.93** |
| 3 | add prompt-group constrained decoding | structural | +0.105 | **+0.000** | **0.00** |

Read the table by *kind*, not by size. The offline gain of #3 is the largest of the three,
and it is worth exactly nothing. The offline gain of #2 is the smallest of the three, and it
is worth almost everything.


In [ ]:
# ---- The three real measurements (aggregate numbers only) -------------------
REAL_POINTS = [
    {"id": 1, "change": "add a word n-gram route to the blend",
     "kind": "feature-side", "d_off": 0.056, "d_lb": 0.025},
    {"id": 2, "change": "add a frozen-encoder linear probe",
     "kind": "feature-side", "d_off": 0.0486, "d_lb": 0.045},
    {"id": 3, "change": "add prompt-group constrained decoding",
     "kind": "structural", "d_off": 0.105, "d_lb": 0.000},
]
REAL = pd.DataFrame(REAL_POINTS)
REAL["r"] = REAL["d_lb"] / REAL["d_off"]
print(REAL.to_string(index=False))

print("\n=== the decision rule this produces ===")
for kind, sub in REAL.groupby("kind"):
    r_mean = float(sub["r"].mean())
    print("%-14s: r = %.2f  ->  an offline gain of +0.05 is worth about %+.4f on the board"
          % (kind, r_mean, r_mean * 0.05))
print("\nRule of thumb from these three points:")
print("  * feature-side change: expect roughly 45-95% of the offline delta to survive")
print("  * structural change: expect ~0, and pay for it with a submission to find out")
print("  * therefore: spend the next submission on a FEATURE-side change,")
print("    and treat a structural change as a hypothesis that only a public A/B can price")


In [ ]:
# ---- Write the artefacts ----------------------------------------------------
summary = {
    "what_this_is": "estimator for r = dLeaderboard / dOffline, and its decision rule",
    "synthetic_protocol": {
        "n_authors": N_AUTHORS, "visible_blocks": N_VIS_BLOCKS,
        "hidden_blocks": N_HID_BLOCKS, "n_worlds": N_WORLDS,
        "scenarios": ["clean", "broken", "swapped"],
        "w_feature_knob": W_FEATURE,
    },
    "transfer_ratio_by_scenario": summary_tab,
    "pooled_estimates": {
        "feature-side": estimate_r([h[0] for h in feat_hist], [h[1] for h in feat_hist])
        if feat_hist else None,
        "structural-side": estimate_r([h[0] for h in struct_hist], [h[1] for h in struct_hist])
        if struct_hist else None,
    },
    "real_measurements_aggregate_only": REAL_POINTS,
    "real_transfer_ratios": {str(r["id"]): float(r["r"]) for _, r in REAL.iterrows()},
    "decision_rule": {
        "feature-side": "expect ~45-95% of the offline delta to survive; safe to buy",
        "structural": "expect ~0; only a public A/B can price it",
    },
}
with open(os.path.join(OUTDIR, "summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2)
print("\nwrote", os.path.join(OUTDIR, "summary.json"))
print("\n=== headline ===")
for kind, sub in REAL.groupby("kind"):
    print("%-14s r = %.2f  (n=%d measured A/B points)" % (kind, sub["r"].mean(), len(sub)))
